# Обучатель-наблюдатель против сетей

`research_m5/cells.py` — один обучатель для любого семейства задач «прочитай последовательность — дай ответ».
Он ищет **маленькую машину**: конечное управление (номер состояния или несколько переключателей) и рядом
до двух чисел (счётчик, уровень с краями, круг). Машину находит решатель SAT; градиентов нет.

**Протокол наблюдателя** (README, этапы 72–74). Ответ виден после каждого символа — как табло автомата,
на котором нажимают кнопки. Учение: 300 последовательностей длиной 1–4 и не больше 40 вопросов; вопрос —
одна последовательность, сколько бы символов в ней ни было. Проверка: 300 последовательностей длиной 20,
100 и 500. Сетям-соперникам то же самое: каждое начало учебных рассказов — свой пример, плюс 40 рассказов
длиной 8–32 на каждые 300.

**Что делает обучатель** (этап 74):

- сначала пробует короткие формы в порядке длины описания в битах;
- потом машины, у которых **состояния — это ответы** (то, что видно), со скрытой ячейкой или несколькими
  скрытыми значениями для того, чего не видно; из объясняющих рассказы берётся та, которой нужно меньше всего
  сказать («иначе ничего не происходит» не говорится);
- **сам спрашивает**: решатель ищет вторую машину, которая тоже объясняет всё услышанное, вместе с рассказом,
  на котором они расходятся; рассказ продолжается обходом ещё не сделанных ходов машины;
- последняя проверка идёт на рассказах до 100 символов; если вопросы кончились раньше или вторая машина не
  исключена, ответ выдаётся с меткой **«не уверен»**.

**Что записано** на 72 открытых семействах (шесть пулов, каждый написан исполнителями, не видевшими
обучателя; 95 % и выше на длине 100):

| пул | 53 | 58 | 60 | 64 | 66 | 70 | всего |
|---|---|---|---|---|---|---|---|
| один выстрел по запечатанному пулу, без наблюдения (этапы 53–70) | 2 | 6 | 7 | 6 | 7 | 3 | 31 из 72 |
| с наблюдением, прежний обучатель (этап 72) | 8 | 9 | 7 | 7 | 7 | 3 | 41 из 72 |
| с наблюдением, этап 74 | 9 | 10 | 8 | 11 | 7 | 6 | **51 из 72** |
| оба обучателя, этап 81 | 10 | 11 | 9 | 11 | 8 | 6 | **55 из 72** |
| оба обучателя, этап 82 (ряды флажков, очередь) | 9 | 12 | 10 | 11 | 10 | 7 | **59 из 72** |
| оба обучателя, этап 85 (стек, целое число со знаком; два семейства упали на ошибке, исправленной после) | 9 | 12 | 9 | 11 | 10 | 6 | 57 из 72 |
| оба обучателя, этап 86 (ранний выход по длине описания; 48 точно и «уверен») | 9 | 12 | 10 | 11 | 11 | 6 | **59 из 72** |
| оба обучателя, этап 89 (проверки в стороне от окончательного ответа и растянутые; 48 точно и «уверен», ложных нет) | 9 | 12 | 10 | 11 | 11 | 6 | **59 из 72** |

На этапе 74 точно (100 / 100 / 100 %) и с меткой «уверен» — 37 семейств; «уверен» и неверно — ни одного.

**Второй обучатель** (этап 81, `parts.py`). Правило, написанное человеком, — чаще не таблица, а несколько
маленьких величин: гости приходят и уходят, дрова кладут и жгут, ночь холодная, когда гости есть, а дров нет.
`parts.py` находит правило в таком виде перебором с проверкой, без решателя: счётчик, уровень с потолком или
круг, который двигают одно–три события; событие — символ вместе с тем, что при нём сделало табло. Число на
табло он читает как сумму и объясняет прибавку. Найденное правило печатается словами. Сначала пробуется он
(`--parts`), и только если он не уверен — машина-таблица (`cells.py`) с оставшимися вопросами.

Записано на тех же 72 семействах (этап 81, оба обучателя): взято **55** вместо 51, точно и «уверен» —
**46** вместо 37, ни одной уверенной машины с ошибкой; 36 семейств берутся правилом из величин за секунды
(медиана 9,5 с). Потеряно два семейства из прежних 51, приобретено шесть.

Все шесть пулов уже открыты: по ним обучатели дорабатывались, так что это не слепая проверка. Слепая —
последняя ячейка: **напишите правило сами**.

Обучатель работает на процессоре; видеокарта нужна только сетям-соперникам.
Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки; все шесть пулов: `%env MX_ALL=1`.

In [ ]:
import os, subprocess
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
CODE = '86ccb87536584ef4dcb9be113ec2467bd6b2146e'  # the code of stage 89 (without the flags of stages 80 and 81 it behaves as recorded at stage 74)
subprocess.run(['git', '-C', ROOT + '/MX-Project', 'checkout', '-q', CODE, '--', 'research_m5/cells.py', 'research_m5/parts.py', 'research_m5/sealed_run.py', 'research_m5/external'], check=True)
!pip -q install python-sat numpy
WORKERS = os.cpu_count()
print('ядер:', WORKERS)

## Обучатель на открытых семействах

Код и флаги — те же, что в записанном прогоне этапа 74. Семейства идут параллельно, по одному на ядро. По
умолчанию — пул 70 (двенадцать семейств, на двух ядрах Colab около получаса–часа: большинство кончается
за минуту-две, несколько упираются в предел 30 минут); с `MX_ALL=1` — все 72. Рядом печатается записанное:
числа могут немного отличаться, потому что предел времени и порог памяти зависят от машины.

In [ ]:
import glob, os, re, subprocess, time
POOLS = ['53', '58', '60', '64', '66', '70'] if os.environ.get('MX_ALL') else ['70']
FLAGS = '--minutes 30 --workers 1 --seconds 3 --shorten --reserve 21 --explore --frugal --slack 16 --guards --build 12 --flags --cascades --keep --deadline --watch --warm --patience 30 --part 100000 --tour 24 --anchor 65 --far 100'
names = [(p, n) for p in POOLS for n in sorted(os.listdir(f'{M5}/sealed/stage{p}_families'))]
if QUICK:
    names = [('53', '05-garden'), ('64', '02-subway-turnstile'), ('58', '08-laundromat'), ('66', '04-hotel-reception')]
groups = [names[i::WORKERS] for i in range(WORKERS)]
os.makedirs('/content/run', exist_ok=True)
procs = []
for g, part in enumerate(groups):
    d = f'/content/run/g{g}'
    os.makedirs(d, exist_ok=True)
    for p, n in part:
        if not os.path.exists(f'{d}/{p}-{n}'):
            os.symlink(f'{M5}/sealed/stage{p}_families/{n}', f'{d}/{p}-{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {FLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/run/out{g}.txt', shell=True))
t0 = time.time()
for p in procs:
    p.wait()
print(f'{time.time() - t0:.0f} s')
mine = {}
for f in glob.glob('/content/run/out*.txt'):
    for line in open(f):
        mine[line.split(' ')[0]] = line.strip()
recorded = {line.split(' ')[0]: line.strip() for line in open(f'{M5}/results/stage74/results.txt')}
acc = lambda s: (re.search(r'100 / 500: [\d.]+ / ([\d.]+)', s) or [None, '?'])[1]
taken = 0
for p, n in names:
    k = f'{p}-{n}'
    line = mine.get(k, k + ': (нет)')
    taken += acc(line) != '?' and float(acc(line)) >= 95
    print(line)
    print('   записано на длине 100:', acc(recorded.get(k, '')), '%', '(не уверен)' if 'NOT SURE' in recorded.get(k, '') else '')
print(f'взято (95 % и выше на длине 100): {taken} из {len(names)}')

## Правило словами (этап 81)

Четыре семейства, которые машина-таблица не брала за 30 минут: горная хижина (18 %), пекарня (22 %), колесо
обозрения (19 %), парковка (56 %). `parts.py` находит каждое за секунды–минуты и печатает правило словами и
программой (`--code`; программа проверяется против найденной машины). Сверьте с кодом автора в `sealed/stage*_families/*/family.py`.

In [ ]:
for p, n in [('60', '09-mountain-hut'), ('58', '01-bakery'), ('60', '11-ferris-wheel'), ('58', '02-parking-garage')][:2 if QUICK else 4]:
    print('=====', n)
    !cd {M5} && OMP_NUM_THREADS=1 python parts.py sealed/stage{p}_families/{n}/family.py --quiet --code

## Оба обучателя вместе

Те же семейства, что в первой ячейке, с флагами `--sum --parts`: сначала правило из величин, потом, если оно
не найдено наверняка, машина-таблица. Рядом — записанное на этапе 89 (`results/stage89/results.txt`) тем же кодом.

In [ ]:
FLAGS81 = FLAGS + ' --sum --parts'
os.makedirs('/content/run81', exist_ok=True)
procs = []
for g, part in enumerate(groups):
    d = f'/content/run81/g{g}'
    os.makedirs(d, exist_ok=True)
    for p, n in part:
        if not os.path.exists(f'{d}/{p}-{n}'):
            os.symlink(f'{M5}/sealed/stage{p}_families/{n}', f'{d}/{p}-{n}')
    procs.append(subprocess.Popen(f'cd {M5} && OMP_NUM_THREADS=1 python sealed_run.py {d} {FLAGS81} 2>/dev/null | grep "^[0-9][0-9]-" > /content/run81/out{g}.txt', shell=True))
for p in procs:
    p.wait()
mine = {}
for f in glob.glob('/content/run81/out*.txt'):
    for line in open(f):
        mine[line.split(' ')[0]] = line.strip()
recorded = {line.split(' ')[0]: line.strip() for line in open(f'{M5}/results/stage89/results.txt') if line[:2].isdigit()}
taken = 0
for p, n in names:
    k = f'{p}-{n}'
    line = mine.get(k, k + ': (нет)')
    taken += acc(line) != '?' and float(acc(line)) >= 95
    print(line[:200])
    print('   записано на длине 100:', acc(recorded.get(k, '')), '%', '(не уверен)' if 'NOT SURE' in recorded.get(k, '') else '')
print(f'взято (95 % и выше на длине 100): {taken} из {len(names)}')

## Другие области: грамматики Томиты, скобки, задачи DeepMind (этапы 83–85)

Те же обучатели — на 19 классических задачах, придуманных задолго до этого проекта
(`worlds/`). Ответ виден после каждого символа, 300 рассказов длиной 1–4, не больше 40 вопросов. Кроме
случайных рассказов — строгая проверка (`worlds/strict.py`): все короткие рассказы и длинные, которые
не сваливаются в ответ, остающийся навсегда (на случайных у скобок почти всегда «сломано», и 100 %
получила бы машина, которая всегда это говорит). Записано на этапе 83: **13 из 13 задач без стека — в
точности**, 12 «уверен», ложной уверенности нет; скобки двух видов (нужен стек) — нет. На этапе 84 в язык
величин добавлен стек: скобки двух и трёх видов и редактор с «отменить» — в точности (скобки трёх видов —
после поправки, найденной при разборе неудачи; это записано как не слепое). На этапе 85 — целое число со
знаком и три новые задачи (a^n b^n, скобки в кавычках, очередь): **18 из 19 в точности**, 17 «уверен»,
ложной уверенности нет; на этапе 86, с бесконечной очередью, — **19 из 19**, 18 «уверен». Затем те же задачи — сетям
на видеокарте, на той же строгой проверке: записано LSTM 6 из 13 на 300 рассказах и 5 на 12 000 (счёт
на длине 500 — скобки, «кто больше», круг — сеть не держит). Здесь можно дать сетям больше шагов
обучения (`STEPS`) и посмотреть трансформер и ALiBi.

In [ ]:
os.makedirs('/content/worlds_found', exist_ok=True)
wn = sorted(d for d in os.listdir(f'{M5}/worlds') if d[:2].isdigit())[:4 if QUICK else 99]
procs = []
for g in range(WORKERS):
    d = f'/content/worldsrun/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in wn[g::WORKERS]:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'{M5}/worlds/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && OMP_NUM_THREADS=1 python sealed_run.py {d} {FLAGS81} --save /content/worlds_found 2>/dev/null | grep "^[0-9][0-9]-" > /content/worldsrun/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cat /content/worldsrun/out*.txt | sort | sed 's/(the commonest[^)]*)//' | cut -c1-170
!cd {M5} && python worlds/strict.py worlds /content/worlds_found

In [ ]:
STEPS = int(os.environ.get('MX_STEPS', '3000'))
kinds = 'lstm' if QUICK else 'lstm transformer alibi'
fams = ' '.join(wn)
!cd {M5} && python cells_rivals.py --folder worlds {fams} --device cuda --watch --strict --steps {STEPS} --kinds {kinds} 2>/dev/null | grep "mean of"

## Чужие наборы: формальные языки из статьи и случайные автоматы (этапы 87–89)

Два набора, которых код не видел, пока его писали.

**15 формальных языков** из работы Bhattamishra, Ahuja, Goyal «On the Ability and Limitations of Transformers
to Recognize Formal Languages» (EMNLP 2020) — та, где трансформеры не обобщают чётность и языки D_n: Shuffle-2/4/6,
Boolean-3/5, a^n b^n c^n, a^n b^n c^n d^n, чётность, (aa)*, (aaaa)*, (abab)*, D_2/3/4/12 (`formal/`). Записаны после
того, как код заморожен (этап 87). Записано: 10 из 15 в точности; на этапе 87 — две ложные уверенности (случайные проверочные рассказы падают в «no» за пару символов), после поправки проверок (этапы 88–89) — ни одной. Не берутся Shuffle-4 и -6 (нужно 5–7 величин), Boolean-5, a^n b^n c^n d^n; a^n b^n c^n — только до n = 11.

**Случайные автоматы**, как в соревновании Abbadingo One (`random_dfa.py`): таблица переходов и принимающие
состояния случайны, 3–8 состояний, 2–3 буквы; их никто не выбирал. Записано вслепую на этапе 89 (зерно 8900): **20 из 20 в точности**, 11 «уверен», ложных нет — из 300 рассказов длиной до 4 и в среднем 30 вопросов.

Ниже — тот же прогон здесь (обучатель на процессоре) и сети на тех же рассказах на видеокарте, со строгой
проверкой: все короткие рассказы и длинные, которые не падают в окончательный ответ.

In [ ]:
!cd {M5} && python random_dfa.py /content/dfa --count 20 --seed 8900
os.makedirs('/content/outside_found', exist_ok=True)
outside = [f'{M5}/formal/{n}' for n in sorted(os.listdir(f'{M5}/formal')) if n[:2].isdigit()] +           [f'/content/dfa/{n}' for n in sorted(os.listdir('/content/dfa'))]
outside = outside[:4] if QUICK else outside
procs = []
for g in range(WORKERS):
    d = f'/content/outsiderun/g{g}'
    os.makedirs(d, exist_ok=True)
    for p in outside[g::WORKERS]:
        if not os.path.exists(f'{d}/{os.path.basename(p)}'):
            os.symlink(p, f'{d}/{os.path.basename(p)}')
    procs.append(subprocess.Popen(f'cd {M5} && OMP_NUM_THREADS=1 python sealed_run.py {d} {FLAGS81} --save /content/outside_found 2>/dev/null | grep "^[0-9][0-9]-" > /content/outsiderun/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cat /content/outsiderun/out*.txt | sort | sed 's/(the commonest[^)]*)//' | cut -c1-170
for g in range(WORKERS):
    !cd {M5} && python worlds/strict.py /content/outsiderun/g{g} /content/outside_found

In [ ]:
kinds = 'lstm' if QUICK else 'lstm transformer'
!cd {M5} && python cells_rivals.py --folder formal --device cuda --watch --strict --steps {STEPS} --kinds {kinds} 2>/dev/null | grep "mean of"
!cd {M5} && python cells_rivals.py --folder /content/dfa --device cuda --watch --strict --steps {STEPS} --kinds {kinds} 2>/dev/null | grep "mean of"

## Соперники на видеокарте

LSTM, трансформер и ALiBi на тех же семействах и по тому же протоколу наблюдателя (`--watch`): каждое
начало учебных рассказов — пример; учение на 300 рассказах и на в 40 раз большем числе, три зерна, 3000
шагов. Засчитывается лучшее. Записанное: на пуле 70 LSTM на 300 рассказах берёт 2 из 12, на 12 000 — 5 из 12; ALiBi на 12 000 (одно зерно) — 2, трансформер — 0; обучатель — 6 из 12 (`results/stage74/rivals_watch_pool70.txt`).

In [ ]:
for p in POOLS if not QUICK else []:
    fams = ' '.join(n for q, n in names if q == p)
    !cd {M5} && python cells_rivals.py --folder sealed/stage{p}_families {fams} --device cuda --watch --kinds lstm transformer alibi 2>/dev/null | grep "mean of"

## Автоматы реальных протоколов (этап 76)

Внешняя проверка: автоматы Мили, которые другие люди сняли с настоящих реализаций (коллекция AALpy,
`external/SOURCE.md`): устройства Bluetooth Low Energy, серверы TLS, брокеры MQTT, стеки TCP. Мерка — найдена ли
**в точности та же машина** (обход по парам состояний, `external/mealy_dot.py --same`).

Записано: замороженным кодом этапа 74 и 40 вопросами — 2 из 23 (неудача по названному заранее порогу); с чтением
ответа «с хода» (`--mealy --reach`) и бюджетом до 200 вопросов — 10 из 23 в точности, от 23 до 140 вопросов на
модель, и две машины «уверен», которые меньше настоящих (серверы NSS и JSSE). Ниже — девять устройств Bluetooth и
пять серверов TLS; модели MQTT и TCP на 15–57 состояний не сходятся и требуют много памяти.

In [ ]:
PFLAGS = FLAGS + ' --deep 24 --mealy --reach --asks 200 --save /content/found'
pnames = sorted(d for d in os.listdir(f'{M5}/external/protocols') if d[:2].isdigit())[:4 if QUICK else 14]
procs = []
for g in range(WORKERS):
    d = f'/content/proto/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in pnames[g::WORKERS]:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'{M5}/external/protocols/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {PFLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/proto/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cat /content/proto/out*.txt | sort | cut -c1-150
!cd {M5} && python external/mealy_dot.py --same /content/found/*.pkl

## Автоматы реальных систем вслепую (этапы 77 и 78)

Вики автоматов университета Радбауд (`automata.cs.ru.nl`): модели, снятые авторами работ с реальных систем.
Отбор — правилом, названным до прогона (`external/wiki_fetch.py`). Записано: первый слепой прогон (70 моделей)
— 38 в точности и 18 ложных «уверен» из 50, неудача по честности; после правила «уверен только когда услышаны
все пары соседних ходов» (`--pairs`) второй слепой прогон на 44 новых моделях — **23 в точности и 2 ложные
уверенности из 21, названный порог взят**. Ячейка повторяет второй прогон: скачивает модели, отбирает, учит и
сверяет в точности.

In [ ]:
!cd {M5} && python external/wiki_fetch.py /content/wiki_raw /content/wiki --beyond 2>/dev/null | tail -n 1
WFLAGS = FLAGS.replace('--minutes 30', '--minutes 15') + ' --deep 24 --mealy --reach --pairs --asks 200 --room 1500 --save /content/found2'
wnames = sorted(d for d in os.listdir('/content/wiki') if d[:2].isdigit())[:4 if QUICK else 44]
procs = []
for g in range(WORKERS):
    d = f'/content/wikirun/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in wnames[g::WORKERS]:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'/content/wiki/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {WFLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/wikirun/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cd {M5} && python external/mealy_dot.py --same /content/found2/*.pkl --in /content/wiki

## Учебные автоматы и автоматы побольше (этапы 79 и 80)

Два слепых прогона, записанных в README, можно повторить здесь. `MX_SET=toys` (по умолчанию) — двенадцать
маленьких автоматов, написанных людьми как примеры, протокол запечатанных пулов (300 рассказов, 40 вопросов):
записано **12 из 12 в точности, ложной уверенности нет**. `MX_SET=larger` — 25 автоматов реальных систем на
11–13 состояний с правилами этапа 80 (`--rare --alike`, 400 вопросов): записано **10 из 25 в точности и 3 ложные
уверенности из 7 — неудача по названному порогу**; этот прогон идёт около полутора часов на четырёх ядрах.

In [ ]:
SET = os.environ.get('MX_SET', 'toys')
!cd {M5} && python external/wiki_fetch.py /content/wiki_raw /content/set_{SET} --{SET} 2>/dev/null | tail -n 1
SFLAGS = FLAGS + ' --deep 24 --mealy --reach --pairs --room 1500 --save /content/found_' + SET +     (' --asks 40' if SET == 'toys' else ' --rare --alike --asks 400')
snames = sorted(d for d in os.listdir(f'/content/set_{SET}') if d[:2].isdigit())[:4 if QUICK else 99]
procs = []
for g in range(WORKERS):
    d = f'/content/setrun_{SET}/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in snames[g::WORKERS]:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'/content/set_{SET}/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {SFLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/setrun_{SET}/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cd {M5} && python external/mealy_dot.py --same /content/found_{SET}/*.pkl --in /content/set_{SET}

## Своё семейство — слепая проверка

Напишите правило сами — обучатель о нём ничего не знает. Нужны `ALPHABET` (2–8 слов) и `answer(seq)`,
которая для любого списка слов (и для пустого) возвращает короткую строку. Найденное правило печатается
словами (величины и что их двигает), если его нашёл `parts.py`, или таблицей. Ниже пример: лифт на пять этажей, который
нельзя сдвинуть, пока открыты двери. Папок с правилами может быть сколько угодно (`02-...`, `03-...`).

In [ ]:
os.makedirs('/content/mine/01-my-family', exist_ok=True)
open('/content/mine/01-my-family/family.py', 'w').write('''
ALPHABET = ["up", "down", "open", "close"]
def answer(seq):
    floor, doors = 0, False
    for t in seq:
        if t == "up" and not doors: floor = min(4, floor + 1)
        elif t == "down" and not doors: floor = max(0, floor - 1)
        elif t == "open": doors = True
        elif t == "close": doors = False
    return f"{floor} {'open' if doors else 'shut'}"
''')
!cd {M5} && OMP_NUM_THREADS=1 python sealed_run.py /content/mine {FLAGS81} --save /content/mine_found 2>/dev/null
!cat /content/mine_found/*.txt /content/mine_found/*.rule.py 2>/dev/null